# Robustez fuera de distribución — Objetivo específico 2

Kong sale muy bien en MAESTRO, pero MAESTRO se grabó con un Disklavier en una sala de concierto. Un usuario real va a subir un MP3 grabado con el celular, con ruido de fondo y el eco del cuarto.

Este notebook mide cuánto pierde el modelo en esas condiciones. Usa las mismas veinte obras del benchmark, las degrada de varias formas y compara contra el audio limpio. Si MAPS está montado como input, también lo evalúa, porque es un piano y una sala distintos a los de MAESTRO.

El resultado decide si el fine-tuning tiene sentido. El objetivo pide una mejora de al menos cinco puntos de F1 de nota sobre el conjunto fuera de distribución. Si la caída total es menor a cinco puntos, la meta no se puede cumplir ni recuperando todo lo perdido.

## 1. Instalación

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("mir_eval")
pip("--no-deps", "piano_transcription_inference", "torchlibrosa")
pip("--no-deps", "pretty_midi", "mido")
pip("librosa", "soundfile")

## 2. Configuración

In [ ]:
import json, glob, random, time
from pathlib import Path
import numpy as np

N_WORKS = 20
SEED = 22779
FRAGMENT_S = 120
FRAGMENT_START_S = 30
ONSET_TOL = 0.05
OFFSET_RATIO = 0.2
SNR_DB = 20
MP3_BITRATE = "64k"
RT60_S = 0.6

DATASET_ROOT = None
for cand in glob.glob("/kaggle/input/*/"):
    hits = glob.glob(cand + "**/maestro-v3.0.0.csv", recursive=True)
    if hits:
        DATASET_ROOT = Path(hits[0]).parent
        break
assert DATASET_ROOT is not None, "Montar the-maestro-dataset-v3-0-0 como input"

RESULTS_DIR = Path("/kaggle/working/robustez")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Obras

Misma semilla que el benchmark, así que son exactamente las mismas veinte obras. De cada una se toman dos minutos a partir del segundo 30, para no caer en silencios del inicio y mantener el costo de GPU bajo.

In [ ]:
import csv

with open(DATASET_ROOT / "maestro-v3.0.0.csv", newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r["split"] == "test"]
works = random.Random(SEED).sample(rows, N_WORKS)

def resolve(rel):
    p = DATASET_ROOT / rel
    return p if p.exists() else Path(glob.glob(str(DATASET_ROOT / "**" / Path(rel).name), recursive=True)[0])

for i, w in enumerate(works, 1):
    print(f"{i:2d}. {w['canonical_composer']} — {w['canonical_title'][:60]}")

## 4. Referencia y métricas

Igual que en el benchmark: los offsets de la referencia se prolongan mientras el pedal está pisado. La única diferencia es que la referencia se recorta al fragmento de dos minutos.

In [ ]:
import pretty_midi
import mir_eval.transcription as mt

def pedal_intervals(pm):
    ccs = sorted((cc for inst in pm.instruments for cc in inst.control_changes if cc.number == 64),
                 key=lambda c: c.time)
    out, down = [], None
    for cc in ccs:
        if cc.value >= 64 and down is None:
            down = cc.time
        elif cc.value < 64 and down is not None:
            out.append((down, cc.time))
            down = None
    if down is not None:
        out.append((down, float("inf")))
    return out

def extend_with_pedal(notes, pedal):
    next_onset, by_pitch = {}, {}
    for n in notes:
        by_pitch.setdefault(n.pitch, []).append(n)
    for group in by_pitch.values():
        group.sort(key=lambda n: n.start)
        for a, b in zip(group, group[1:]):
            next_onset[id(a)] = b.start
    out = []
    for n in notes:
        end = n.end
        for s, r in pedal:
            if s <= end < r:
                end = r
                break
        end = min(end, next_onset.get(id(n), float("inf")))
        out.append((n.start, max(end, n.start + 1e-6), n.pitch))
    return out

def load_reference(midi_path, start=0.0, duration=None):
    pm = pretty_midi.PrettyMIDI(str(midi_path))
    notes = [n for inst in pm.instruments if not inst.is_drum for n in inst.notes]
    notes = extend_with_pedal(notes, pedal_intervals(pm))
    end = start + duration if duration else float("inf")
    notes = [(s - start, min(e, end) - start, p) for s, e, p in notes if start <= s < end]
    notes.sort()
    intervals = np.array([[s, e] for s, e, _ in notes])
    pitches = np.array([pretty_midi.note_number_to_hz(p) for _, _, p in notes])
    return intervals, pitches

def evaluate(ref_i, ref_p, est_i, est_p):
    if len(est_i) == 0:
        return {"f1_onset": 0.0, "f1_note": 0.0, "ner": 1.0}
    _, _, f_on, _ = mt.precision_recall_f1_overlap(
        ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL, offset_ratio=None)
    _, _, f_note, _ = mt.precision_recall_f1_overlap(
        ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL,
        offset_ratio=OFFSET_RATIO, offset_min_tolerance=0.05)
    tp = len(mt.match_notes(ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL, offset_ratio=None))
    ner = (len(ref_i) - tp + len(est_i) - tp) / len(ref_i)
    return {"f1_onset": float(f_on), "f1_note": float(f_note), "ner": float(ner)}

## 5. Degradaciones

Cada condición imita algo que pasa de verdad al grabar en casa:

| Condición | Qué simula |
|---|---|
| `limpio` | El audio original, sirve de línea base |
| `mp3` | Compresión con pérdida a 64 kbps, como un archivo mandado por WhatsApp |
| `ruido` | Ruido de fondo a 20 dB de SNR |
| `reverb` | El eco de un cuarto cerrado, con un RT60 de 0.6 s |
| `celular` | Las tres anteriores juntas, que es el caso más realista |

La reverberación usa una respuesta al impulso sintética de ruido con caída exponencial. No es una sala real, pero reproduce la cola que alarga las notas y confunde la detección de offsets.

In [ ]:
import soundfile as sf
import librosa
from scipy.signal import fftconvolve

rng = np.random.default_rng(SEED)

def add_noise(x, snr_db=SNR_DB):
    noise = rng.standard_normal(len(x))
    scale = np.sqrt(np.mean(x ** 2) / (10 ** (snr_db / 10) * np.mean(noise ** 2)))
    return x + scale * noise

def add_reverb(x, sr, rt60=RT60_S):
    t = np.arange(int(rt60 * sr)) / sr
    ir = rng.standard_normal(len(t)) * np.exp(-6.9 * t / rt60)
    ir[0] = 1.0
    y = fftconvolve(x, ir)[: len(x)]
    return y / (np.max(np.abs(y)) + 1e-9) * np.max(np.abs(x))

def mp3_roundtrip(x, sr, bitrate=MP3_BITRATE):
    sf.write("/tmp/_in.wav", x, sr)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/tmp/_in.wav",
                    "-b:a", bitrate, "/tmp/_out.mp3"], check=True)
    y, _ = librosa.load("/tmp/_out.mp3", sr=sr, mono=True)
    return y[: len(x)]

CONDITIONS = {
    "limpio": lambda x, sr: x,
    "mp3": mp3_roundtrip,
    "ruido": lambda x, sr: add_noise(x),
    "reverb": add_reverb,
    "celular": lambda x, sr: mp3_roundtrip(add_noise(add_reverb(x, sr)), sr),
}

## 6. Modelo

In [ ]:
import torch
from piano_transcription_inference import PianoTranscription, sample_rate as SR

model = PianoTranscription(device="cuda" if torch.cuda.is_available() else "cpu")

def transcribe(audio):
    out = model.transcribe(audio, "/tmp/_kong.mid")
    return [(float(e["onset_time"]), float(e["offset_time"]), int(e["midi_note"]))
            for e in out["est_note_events"]]

def to_arrays(notes):
    if not notes:
        return np.zeros((0, 2)), np.zeros(0)
    return (np.array([[s, e] for s, e, _ in notes]),
            np.array([pretty_midi.note_number_to_hz(p) for _, _, p in notes]))

## 7. Ejecución

Un JSON por obra y condición, con las notas estimadas guardadas. Si la sesión se cae, al volver a correr solo se procesa lo que falta.

In [ ]:
def run(source, key, audio_loader, ref_loader):
    for cond, degrade in CONDITIONS.items():
        out = RESULTS_DIR / f"{source}__{cond}__{key}.json"
        if out.exists():
            continue
        x = degrade(audio_loader(), SR)
        t0 = time.time()
        est = transcribe(x)
        m = evaluate(*ref_loader(), *to_arrays(est))
        m.update({"source": source, "condition": cond, "work": key,
                  "latency_norm": (time.time() - t0) / (len(x) / SR), "est_notes": est})
        out.write_text(json.dumps(m))
        print(f"[{source}/{cond:8s}] {key[:40]:40s} F1on={m['f1_onset']:.3f} F1note={m['f1_note']:.3f}")

for w in works:
    audio_path, midi_path = resolve(w["audio_filename"]), resolve(w["midi_filename"])
    run("maestro", Path(w["midi_filename"]).stem,
        lambda: librosa.load(str(audio_path), sr=SR, mono=True,
                             offset=FRAGMENT_START_S, duration=FRAGMENT_S)[0],
        lambda: load_reference(midi_path, FRAGMENT_START_S, FRAGMENT_S))

## 8. MAPS

MAPS no está en Kaggle de forma oficial. Si lo montaste como input, esta celda busca las grabaciones reales del Disklavier (`ENSTDkCl` con micrófono cercano y `ENSTDkAm` con micrófono ambiente) y las evalúa con las mismas condiciones. Si no lo encuentra, la sigue de largo.

In [ ]:
maps_wavs = sorted(p for p in glob.glob("/kaggle/input/**/ENSTDk*/MUS/*.wav", recursive=True))
print(f"Grabaciones de MAPS encontradas: {len(maps_wavs)}")

for wav in maps_wavs:
    wav = Path(wav)
    run("maps_" + wav.parent.parent.name, wav.stem,
        lambda: librosa.load(str(wav), sr=SR, mono=True, duration=FRAGMENT_S)[0],
        lambda: load_reference(wav.with_suffix(".mid"), 0.0, FRAGMENT_S))

## 9. Resultados

In [ ]:
import pandas as pd

records = [json.loads(p.read_text()) for p in RESULTS_DIR.glob("*.json")]
df = pd.DataFrame(records).drop(columns="est_notes")

agg = (df.groupby(["source", "condition"])[["f1_onset", "f1_note", "ner", "latency_norm"]]
         .mean().mul([100, 100, 100, 1]).round(2))
base = agg.loc[("maestro", "limpio"), "f1_note"]
agg["caida_f1_note"] = (agg["f1_note"] - base).round(2)
agg

## 10. Cuánto pesa el offset en la partitura

El F1 de nota es la métrica más baja porque exige acertar también el final de cada nota. La pregunta es si ese error llega al BRF. El cuantizador redondea cada duración a una grilla de semicorchea, así que un offset que se equivoca por pocos milisegundos termina en la misma figura.

Para medirlo se toman las notas que el modelo acertó por onset y altura, se cuantizan la duración de la referencia y la estimada, y se cuenta cuántas veces coinciden. El tempo real de cada obra no está en MAESTRO, así que se prueba con 60 y 120 bpm, que cubren el rango de tempos del repertorio.

Con esto quedan tres números por condición: el porcentaje de offsets correctos según `mir_eval`, y el porcentaje de figuras correctas después de cuantizar a cada tempo.

In [ ]:
def ticks(dur_s, bpm):
    return min(16, max(1, round(dur_s / (60 / bpm / 4))))

rows = []
for r in records:
    if r["source"] != "maestro":
        continue
    w = next(w for w in works if Path(w["midi_filename"]).stem == r["work"])
    ref_i, ref_p = load_reference(resolve(w["midi_filename"]), FRAGMENT_START_S, FRAGMENT_S)
    est_i, est_p = to_arrays([tuple(n) for n in r["est_notes"]])
    pairs = mt.match_notes(ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL, offset_ratio=None)
    if not pairs:
        continue
    ref_d = np.array([ref_i[i, 1] - ref_i[i, 0] for i, _ in pairs])
    est_d = np.array([est_i[j, 1] - est_i[j, 0] for _, j in pairs])
    tol = np.maximum(0.05, OFFSET_RATIO * ref_d)
    row = {"condition": r["condition"], "offset_ok": np.mean(np.abs(ref_d - est_d) <= tol)}
    for bpm in (60, 120):
        row[f"figura_ok_{bpm}bpm"] = np.mean([ticks(a, bpm) == ticks(b, bpm) for a, b in zip(ref_d, est_d)])
    rows.append(row)

(pd.DataFrame(rows).groupby("condition").mean().mul(100).round(2)
   .reindex(list(CONDITIONS)))

## 11. Veredicto

In [ ]:
maestro = agg.xs("maestro", level="source")
ood = agg.drop(index=("maestro", "limpio"))
peor = ood["caida_f1_note"].min()

print(f"F1 de nota en MAESTRO limpio: {maestro.loc['limpio', 'f1_note']:.2f} %")
print(f"Peor caída fuera de distribución: {peor:.2f} puntos")
print()
if -peor < 5:
    print("La caída es menor a 5 puntos. Un fine-tuning no puede cumplir la meta del objetivo")
    print("porque no hay cinco puntos que recuperar.")
else:
    print("La caída supera los 5 puntos. Hay margen para que el fine-tuning cumpla la meta.")

### Observaciones

*(Completar al correr: cuánto cae cada condición, cuál afecta más, si el offset cambia la figura escrita y qué implica para el objetivo 2.)*